In [1]:
# ==========================================
# CELL 1: Install Dependencies & Setup Environment
# ==========================================
!pip install -q "pillow<11.1.0"
!pip install -q -U transformers accelerate bitsandbytes datasets faiss-cpu langchain langchain-community langchain-huggingface sentence-transformers pandas

import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("Classic")
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 74.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ==========================================
# CELL 2: Load Instruction-Tuned MedGemma Safely (CPU-First)
# ==========================================
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "google/medgemma-1.5-4b-it"

print(f"Loading processor and conversational model: {model_id}...")
processor = AutoProcessor.from_pretrained(model_id, token=hf_token)

print("Loading model weights onto CPU memory first...")
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
    device_map=None,
    token=hf_token
)

device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    if device == "cuda":
        print("Shifting conversational model to CUDA...")
        model = model.to(torch.bfloat16).to("cuda")
        print("Model successfully moved to GPU!")
    else:
        print("Using CPU mode.")
except Exception as e:
    print(f"GPU shift skipped ({e}). Staying on CPU.")
    model = model.to("cpu")

print("MedGemma Chat Assistant is ready!")

Loading processor and conversational model: google/medgemma-1.5-4b-it...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Shifting conversational model to CUDA...
Model successfully moved to GPU!
MedGemma Chat Assistant is ready!


In [3]:
# ==========================================
# CELL 3: Build RAG Vector Store from BraTS Clinical CSV QA Pairs
# ==========================================
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Update path to point to your uploaded CSV file location in Kaggle input
csv_path = "/kaggle/input/datasets/aliqaiser1123/brats-qa-pairs/brats_clinical_qa_4_features (1).csv"

print(f"Loading clinical QA dataset from: {csv_path}")
df = pd.read_csv(csv_path)

# Convert rows into rich context document strings for vector indexing
medical_knowledge_base = []
for _, row in df.iterrows():
    doc_content = (
        f"Patient ID: {row['patient_id']} | "
        f"Clinical Question: {row['question']} | "
        f"Verified Medical Answer: {row['answer']}"
    )
    medical_knowledge_base.append(Document(page_content=doc_content))

print(f"Loaded {len(medical_knowledge_base)} expert clinical QA records into memory.")

print("Initializing embeddings on CPU...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = FAISS.from_documents(medical_knowledge_base, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Advanced BraTS RAG Vector Store Index Ready.")

/tmp/ipykernel_23/2711000709.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading clinical QA dataset from: /kaggle/input/datasets/aliqaiser1123/brats-qa-pairs/brats_clinical_qa_4_features (1).csv
Loaded 4992 expert clinical QA records into memory.
Initializing embeddings on CPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Advanced BraTS RAG Vector Store Index Ready.


In [4]:
# ==========================================
# CELL 4: Multi-Turn Conversational RAG Engine (Forced CPU Safe Inference)
# ==========================================
from PIL import Image

chat_history = [
    {
        "role": "system",
        "content": (
            "You are a compassionate, clinical neuro-oncology assistant chatbot. "
            "Your job is to converse with patients and answer questions accurately using the provided RAG context. "
            "Never hallucinate tumor specifications, sizes, or spatial locations."
        )
    }
]

def chat_with_patient(image_path, user_message):
    # 1. RAG Retrieval from CSV Clinical Dataset
    retrieved_docs = retriever.invoke(user_message)
    rag_context = "\n".join([doc.page_content for doc in retrieved_docs]) if retrieved_docs else "General clinical data."

    enhanced_message = (
        f"[Verified BraTS Clinical Database Match]:\n{rag_context}\n\n"
        f"[Patient/User Message]: {user_message}"
    )

    # 2. Attach image if provided
    if image_path and os.path.exists(image_path):
        image = Image.open(image_path).convert("RGB")
        content_payload = [{"type": "image", "image": image}, {"type": "text", "text": enhanced_message}]
    else:
        content_payload = [{"type": "text", "text": enhanced_message}]

    chat_history.append({"role": "user", "content": content_payload})

    # 3. Format prompt using official chat template
    prompt_text = processor.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)

    active_images = [
        item["image"] for turn in chat_history if isinstance(turn["content"], list)
        for item in turn["content"] if item.get("type") == "image"
    ]

    # Force inputs and model onto CPU to bypass hardware kernel mismatch errors
    inputs = processor(
        text=[prompt_text],
        images=active_images if active_images else None,
        return_tensors="pt"
    ).to("cpu")

    model.to("cpu")

    # 4. Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=True,
            temperature=0.2
        )

    input_token_length = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0][input_token_length:]
    response_text = processor.decode(generated_tokens, skip_special_tokens=True)

    chat_history.append({"role": "assistant", "content": [{"type": "text", "text": response_text}]})

    return response_text

print("Conversational RAG pipeline compiled successfully.")

Conversational RAG pipeline compiled successfully.


In [5]:
# ==========================================
# CELL 5: Test Live Patient Chat Session
# ==========================================
# Point to a specific patient slice from your extracted tar or dataset folder
sample_img_path = "/kaggle/input/brats-2021-task-1-dataset/BraTS2021_00495/BraTS2021_00495_t1ce.nii.gz" # Or use a sample slice path if converted to jpg

print("--- Turn 1: Asking about patient tumor size ---")
reply_1 = chat_with_patient(None, "What is the total size of the tumor for BraTS2021_00063?")
print(f"Assistant: {reply_1}\n" + "-"*50)

print("--- Turn 2: Follow-up asking about sub-region space ---")
reply_2 = chat_with_patient(None, "Can you describe the sub-region space breakdown for that tumor?")
print(f"Assistant: {reply_2}\n" + "-"*50)

--- Turn 1: Asking about patient tumor size ---
Assistant: Based on the provided clinical database, the total tumor volume for BraTS2021_00063 is 31569 mm³.
--------------------------------------------------
--- Turn 2: Follow-up asking about sub-region space ---
Assistant: <unused94>thought
1.  **Identify the core request:** The user wants to know the sub-region space breakdown (necrotic core, peritumoral edema, enhancing tumor) for a specific tumor ID.

2.  **Identify the specific tumor ID:** The user's message doesn't explicitly state a tumor ID. However, the previous question was about "BraTS2021_00063". It's highly probable the user is asking about the *same* tumor.

3.  **Access the knowledge base:** Search the provided RAG context for the tumor ID "BraTS2021_00063".

4.  **Find the matching entry:** Locate the entry:
    `Patient ID: BraTS2021_00063 | Clinical Question: Describe the subregion space of the tumor. | Verified Medical Answer: The tumor comprises 10805 mm³ of necroti

In [6]:
# ==========================================
# CLEAN EVALUATION CELL (FIXED)
# ==========================================
!pip install -q bert-score rouge-score

import pandas as pd
import numpy as np
from bert_score import score as bert_score
from rouge_score import rouge_scorer

# 1. Ensure the DataFrame is loaded in this scope
csv_path = "/kaggle/input/datasets/aliqaiser1123/brats-qa-pairs/brats_clinical_qa_4_features (1).csv"
eval_df = pd.read_csv(csv_path)

def clean_output(response_text):
    """Removes the <unused94>thought...<unused95> block from the output."""
    if "<unused95>" in response_text:
        return response_text.split("<unused95>")[-1].strip()
    return response_text

def evaluate_all_four_features(patient_id, df):
    # Filter records using the passed dataframe
    patient_records = df[df['patient_id'] == patient_id]
    results = []
    
    print(f"\n{'='*10} CLINICAL ACCURACY REPORT: {patient_id} {'='*10}")
    
    if patient_records.empty:
        print(f"No records found for patient {patient_id}.")
        return

    for _, row in patient_records.iterrows():
        query, reference = row['question'], row['answer']
        
        # Generate & Clean Response
        raw_response = chat_with_patient(None, f"For {patient_id}, {query}")
        model_response = clean_output(raw_response)
        
        # Calculate Metrics
        is_faithful = any(word in model_response for word in reference.split() if len(word) > 4)
        _, _, f1 = bert_score([model_response], [reference], lang="en", verbose=False, device="cpu")
        bert_f1 = f1.mean().item()
        
        results.append({'bert': bert_f1, 'faith': is_faithful})
        
        # Print Clean Results
        print(f"\nQ: {query}")
        print(f"✅ Expected: {reference}")
        print(f"🤖 Received: {model_response}")
        print(f"🔍 Faithfulness: {'PASS' if is_faithful else 'FAIL'} | BERT-F1: {bert_f1:.2f}")

    # Summary Statistics
    avg_bert = np.mean([r['bert'] for r in results])
    print(f"\n{'='*10} SUMMARY {'='*10}")
    print(f"Average Semantic Accuracy (BERT): {avg_bert:.2f}")
    print("Status: " + ("EXCELLENT" if avg_bert > 0.8 else "GOOD/FAIR"))

# --- Run the evaluation ---
evaluate_all_four_features("BraTS2021_00063", eval_df)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00

========== CLINICAL ACCURACY REPORT: BraTS2021_00063 ==========


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Q: What is the spatial location of the tumor in the brain?
✅ Expected: The tumor is primarily located in the right, posterior, and superior region of the brain.
🤖 Received: Based on the provided clinical database
🔍 Faithfulness: FAIL | BERT-F1: 0.85


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Q: Is the tumor multifocal?
✅ Expected: No, the tumor is unifocal, presenting as a single contiguous mass.
🤖 Received: Based on the provided clinical database
🔍 Faithfulness: FAIL | BERT-F1: 0.85

========== SUMMARY ==========
Average Semantic Accuracy (BERT): 0.86
Status: EXCELLENT
